# Multitemporal LULC Analysis Proof of Concept: Model Transfer & Batch Mode

This notebook serves as a live Google Earth Engine proof-of-concept for **Multitemporal Land Cover Mapping (`N >= 2` periods)** and **Post-Classification Consistency Analysis (`ConsistencyAnalyzer`)** across both supported workflows:

1. **Workflow A — Model Transfer Mode (`1` training dataset on `baseline`)**:
   - Acquires and harmonizes imagery across `historical` and `baseline` (with optional BRDF, SCSc terrain correction, and CDF histogram matching).
   - Runs separability analysis (`TD`/`JM` + optional post-predictor `SILHOUETTE`) on `baseline` only.
   - Computes a unified predictor stack across all periods (spectral indices, static terrain + `landform`, static distance `dist_*`, and period-matched SAR) with optional `RFE_CV` feature selection.
   - Extracts features once on `baseline` (`extraction_type='single'`), trains a `MULTIPROBABILITY` Random Forest (with optional Gaussian spatial smoothing and `log_loss`), and transfers the trained model to `historical`.
   - Runs **HMM / Viterbi Post-Classification Consistency Analysis** anchored to `baseline`.

2. **Workflow B — Batch Mode (`2` training datasets: `historical` + `baseline`)**:
   - Accepts separate training data (`historical_roi` and `baseline_roi`) and validation data (`historical_val` and `baseline_val`).
   - Runs separability analysis independently on **both** `historical` and `baseline` (never mixed).
   - Uses the same unified predictor set (and optional unified `RFE_CV` selection) across all periods.
   - Performs batch feature extraction (`extraction_type='batch'`), trains and evaluates independent Random Forest models per period, and runs batch thematic accuracy assessment across both validation sets.
   - Applies **HMM / Joint Viterbi** & **Rule-Based** consistency correction and compares pre- vs. post-correction thematic accuracy (`compare_accuracy`).

## 0. Setup, Earth Engine Initialization & Shared Inputs

In [1]:
import os
import sys
import ee
import geemap
import pandas as pd
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
src_path = os.path.join(repo_root, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import luma_ge
from luma_ge import PeriodConfig, MultitemporalAnalysis, consistency_analysis
from luma_ge.predictor import predictor_catalog
from luma_ge.accuracy import thematic_accuracy

assert luma_ge.auto_initialize(), 'Earth Engine initialization failed!'
print('Earth Engine initialized:', luma_ge.get_auth_status())

2026-11-24 15:57:18,727 - luma_ge.ee_config - INFO - Attempting to initialize Earth Engine with service account for project: epistem-luma-launching


DEBUG: Starting auto_initialize...
DEBUG: All environment variables containing 'GOOGLE': []
DEBUG: Checking environment variables...
DEBUG: GOOGLE_SERVICE_ACCOUNT_JSON exists: False
DEBUG: GOOGLE_APPLICATION_CREDENTIALS: None
DEBUG: Checking auth directory: c:\Users\AFahrezi\Documents\GitHub\luma-stack\auth
DEBUG: Auth directory exists, listing files...
DEBUG: Found file in auth/: epistem-luma-launching-92541db9962d.json
DEBUG: Using auth file: c:\Users\AFahrezi\Documents\GitHub\luma-stack\auth\epistem-luma-launching-92541db9962d.json
DEBUG: Found service account file: c:\Users\AFahrezi\Documents\GitHub\luma-stack\auth\epistem-luma-launching-92541db9962d.json


2026-11-24 15:57:20,430 - googleapiclient.http - WARNING - Encountered 403 Forbidden with reason "PERMISSION_DENIED"
2026-11-24 15:57:20,432 - luma_ge.ee_config - ERROR - Service account initialization failed: Caller does not have required permission to use project epistem-luma-launching. Grant the caller the roles/serviceusage.serviceUsageConsumer role, or a custom role with the serviceusage.services.use permission, by visiting https://console.developers.google.com/iam-admin/iam?project=epistem-luma-launching and then retry. Propagation of the new permission may take a few minutes.
2026-11-24 15:57:20,432 - luma_ge.ee_config - INFO - Trying alternative authentication method...
2026-11-24 15:57:22,945 - luma_ge.ee_config - INFO - Earth Engine initialized with alternative method for project: epistem-luma-launching


Earth Engine initialized with service account: c:\Users\AFahrezi\Documents\GitHub\luma-stack\auth\epistem-luma-launching-92541db9962d.json
Earth Engine initialized: {'initialized': True, 'authenticated': True, 'project': None, 'user_info': None}


In [ ]:
#1. Shared AOI & Classification Scheme
#testing sites: Malaka Tengah
aoi = ee.FeatureCollection('projects/python-ee-487304/assets/AOI_Malaka_Tengah')
scheme_df = pd.read_csv(os.path.join(repo_root, 'data', 'MODUL2_Skema_Klasifikasi_Malaka_Tengah.csv'))
display(scheme_df)
#2. Training & Validation Data (thematic)
baseline_roi = ee.FeatureCollection('projects/python-ee-487304/assets/Training_Malaka_2025')
baseline_val = ee.FeatureCollection('projects/python-ee-487304/assets/Test_Malaka_2025')
historical_roi = ee.FeatureCollection('projects/python-ee-487304/assets/Training_Malaka_2015')
historical_val = ee.FeatureCollection('projects/python-ee-487304/assets/Test_Malaka_2015')

,ID,Land Cover Class,Color Palette
0,1,Hutan Lahan Kering,#006400
1,2,Hutan Mangrove,#759237
2,3,Jati Monokultur,#5840a0
3,4,Kebun Campur,#d534ff
4,5,Padang Rumput,#A7FF71
5,6,Pertanian Lahan Kering,#FFC500
6,7,Sawah,#FFFD00
7,8,Lahan Terbangun,#ffa9eb
8,9,Lahan Terbuka,#ff1f1f
9,10,Tambak,#9ecaff


## 1. Period Definitions & Multitemporal Predictor Availability Check 
Before selecting predictors, `predictor_catalog.check_multitemporal_availability()` checks which spectral, terrain, distance, and SAR predictors are valid across **every** configured period.

In [3]:
#define configuration(historical vs baseline, dates, and sensors)
periods_config = [
    PeriodConfig(label='historical', start_date=2015, end_date=2015, sensor='L8_SR',     cloud_cover=30),
    PeriodConfig(label='baseline',   start_date=2025, end_date=2025, sensor='L9_SR',     cloud_cover=30),
]
#check predictor
availability = predictor_catalog.check_multitemporal_availability(periods_config)
avail_df = pd.DataFrame.from_dict(availability, orient='index')[
    ['category', 'type', 'available', 'unavailable_periods', 'reason']
]
print(f"Available predictors across all periods: {avail_df['available'].sum()} / {len(avail_df)}")
display(avail_df)

Available predictors across all periods: 23 / 31


,category,type,available,unavailable_periods,reason
NDVI,Vegetasi,spectral,True,[],
EVI,Vegetasi,spectral,True,[],
SAVI,Vegetasi,spectral,True,[],
MSAVI,Vegetasi,spectral,True,[],
GNDVI,Vegetasi,spectral,True,[],
MCARI,Vegetasi,spectral,False,"[historical, baseline]",Requires Sentinel-2 Red-Edge band (RE1) (unava...
RENDVI,Vegetasi,spectral,False,"[historical, baseline]","Requires Sentinel-2 Red-Edge bands (RE1, RE2) ..."
MCARI1,Vegetasi,spectral,True,[],
MNDWI,Air & Kelembaban,spectral,True,[],
NDMI,Air & Kelembaban,spectral,True,[],


In [4]:
# Shared predictor configuration used by both Model Transfer and Batch workflows
# Static predictors (elevation, slope, landform, dist_*) are computed ONCE per AOI;
# dynamic predictors (spectral indices, Sentinel-1 / PALSAR) are computed per period.
shared_predictor_config = {
    'individual_predictors': {
        'elevation': True,
        'slope': True,
        'aspect': False,
        'landform': True,
        'distance': ['dist_roads'],
        'spectral_indices': ['NDVI', 'MSAVI', 'MCARI1', 'ANDWI', 'NDBI', 'DBSI'],
    }
}

# Optional RFE_CV configuration (Module 5): set to None to skip RFE_CV, or pass a dict
shared_rfe_config = {
    'enabled': True,
    'min_features': 6,
    'step': 1,
    'cv': 3,
    'n_trees': 50,
    'scoring': 'accuracy',
    'per_period_rfe': False,  # False = unified predictor subset selected once on reference_period
}

---
## 2. Workflow A: Model Transfer Mode (`1` Training Dataset on `baseline`)
When the user provides training data for **`baseline` only** (`training_sources={'historical': None, 'baseline': baseline_roi}`):
- **Module 3 (`load_training_data`)** detects `mode = 'model_transfer'`.
- **Module 4 (`check_separability`)** runs separability (`TD`/`JM` + optional `SILHOUETTE`) on `baseline` only.
- **Module 5 (`compute_predictors` + optional `run_feature_selection`)** builds the harmonized predictor stack for all periods and optionally selects the optimal predictor subset via `RFE_CV`.
- **Module 6 (`extract_features` + `run_classification`)** extracts samples once on `baseline` (`extraction_type = 'single'`), trains the `baseline` Random Forest (`MULTIPROBABILITY`), optionally applies simple Gaussian spatial smoothing on `prob_*` before `argmax`, and transfers the model to `historical`.
- **Module 7 (`assess_accuracy`)** evaluates thematic accuracy on `baseline_val` (and `historical_val` if provided).

In [ ]:
mt_transfer = MultitemporalAnalysis(
    aoi=aoi,
    periods=periods_config,
    reference_period='baseline',
    apply_brdf=True,            
    apply_topo_correction=False, # Set True to enable SCSc topographic correction
    histogram_match=True,        # Match historical composite CDF onto baseline
    scale=30,
)

res_transfer = mt_transfer.run_pipeline(
    training_sources={'historical': None, 'baseline': baseline_roi},
    landcover_df=scheme_df,
    predictor_config=shared_predictor_config,
    validation_by_period={'historical': None, 'baseline': baseline_val},
    class_field='ID_Kelas',
    class_name_property='KelasTupla',
    use_class_ids=True,
    min_samples=5,
    run_separability=True,
    include_silhouette=True,     # Post-predictor non-parametric Silhouette score
    rfe_config=shared_rfe_config,
    classification_type='soft',
    soft_mode='multiprobability',
    ntrees=150,
    include_log_loss=True,
    spatial_smoothing=True,      # Optional simple Gaussian smoothing on prob_* before argmax
    smoothing_radius=1,
    smoothing_sigma=1.0,
    verbose=True,
)

eval_tr_base = res_transfer['classification']['model_evaluation']['baseline']
print('=== Workflow A (Model Transfer) Summary ===')
print('Detected mode:', res_transfer['mode'])
print('Separability periods analyzed:', res_transfer['separability']['analyzed_periods'])
print('Separability periods skipped:', res_transfer['separability']['skipped_periods'])
if res_transfer.get('rfe'):
    print('RFE_CV selected features:', res_transfer['rfe']['selected_features'])
print('Feature extraction type:', res_transfer['feature_extraction']['extraction_type'])
print('Baseline internal test OA:', eval_tr_base['overall_accuracy'], '| Log Loss:', eval_tr_base.get('log_loss'))
if res_transfer.get('accuracy') and 'baseline' in res_transfer['accuracy']['assessed_periods']:
    ok, acc_tr_base = res_transfer['accuracy']['per_period']['baseline']
    if ok:
        print('Baseline external validation OA:', acc_tr_base['overall_accuracy'], '| Kappa:', acc_tr_base['kappa'])

2026-11-24 15:57:48,515 - final_Image - INFO - final_Image creation initialized.
2026-11-24 15:57:48,516 - Reflectance_Data - INFO - ReflectanceData initialized.
2026-11-24 15:57:48,545 - BRDF_Correction - INFO - BRDF_Correction initialised.
2026-11-24 15:57:48,547 - Reflectance_Data - INFO - Starting data fetch for Landsat 8 Operational Land Imager Surface Reflectance
2026-11-24 15:57:48,547 - Reflectance_Data - INFO - Date range: 2015-01-01 to 2015-12-31
2026-11-24 15:57:48,548 - Reflectance_Data - INFO - Cloud cover threshold: 30%
2026-11-24 15:57:48,549 - Reflectance_Stats - INFO - Reflectance Stats initialized.
2026-11-24 15:57:52,031 - Reflectance_Data - INFO - Initial collection (before cloud filtering): 42 images
2026-11-24 15:57:52,033 - Reflectance_Data - INFO - Date range of available images: 2015-01-15 to 2015-12-17
2026-11-24 15:57:52,979 - Reflectance_Data - INFO - After cloud filtering (<30%): 32 images
2026-11-24 15:57:52,981 - Reflectance_Data - INFO - Cloud cover of s

Stratified Random Split Training Pixel Size: 360


2026-11-24 15:58:10,529 - luma_ge.classification - INFO - Multitemporal feature extraction (single / model_transfer): extracted=['baseline'], skipped=['historical']


Stratified Random Split Testing Pixel Size: 175


Extracting GEE samples: 0batch [00:00, ?batch/s]

2026-11-24 16:01:06,164 - luma_ge.predictor - INFO - Starting local RFECV on 359 samples, 18 features (min_features=6, cv=3, n_trees=50)


Starting local RFECV on 359 samples, 18 features (min_features=6, cv=3, n_trees=50)


Running local RFECV:   0%|          | 0/53 [00:00<?, ?fit/s]

2026-11-24 16:01:12,886 - luma_ge.predictor - INFO - Recommended feature stack (16 features):
2026-11-24 16:01:12,887 - luma_ge.predictor - INFO -   - BLUE
2026-11-24 16:01:12,888 - luma_ge.predictor - INFO -   - GREEN
2026-11-24 16:01:12,890 - luma_ge.predictor - INFO -   - RED
2026-11-24 16:01:12,892 - luma_ge.predictor - INFO -   - NIR
2026-11-24 16:01:12,894 - luma_ge.predictor - INFO -   - SWIR1
2026-11-24 16:01:12,895 - luma_ge.predictor - INFO -   - SWIR2
2026-11-24 16:01:12,897 - luma_ge.predictor - INFO -   - elevation
2026-11-24 16:01:12,898 - luma_ge.predictor - INFO -   - dist_roads
2026-11-24 16:01:12,900 - luma_ge.predictor - INFO -   - dist_coast
2026-11-24 16:01:12,900 - luma_ge.predictor - INFO -   - dist_settlement
2026-11-24 16:01:12,901 - luma_ge.predictor - INFO -   - NDVI
2026-11-24 16:01:12,902 - luma_ge.predictor - INFO -   - MSAVI
2026-11-24 16:01:12,903 - luma_ge.predictor - INFO -   - MCARI1
2026-11-24 16:01:12,904 - luma_ge.predictor - INFO -   - ANDWI
2026-

Recommended feature stack (16 features):
  - BLUE
  - GREEN
  - RED
  - NIR
  - SWIR1
  - SWIR2
  - elevation
  - dist_roads
  - dist_coast
  - dist_settlement
  - NDVI
  - MSAVI
  - MCARI1
  - ANDWI
  - NDBI
  - DBSI


---
## 3. Workflow B: Batch Mode (`2` Training Datasets: `historical` + `baseline`)
When the user provides training data for **both `historical` and `baseline`** (`training_sources={'historical': historical_roi, 'baseline': baseline_roi}`):
- **Module 3 (`load_training_data`)** detects `mode = 'batch'`.
- **Module 4 (`check_separability`)** runs separability analysis on **both** `historical` and `baseline` independently (never mixed).
- **Module 5 (`compute_predictors` + optional `run_feature_selection`)** applies the same predictor configuration across both periods and (with `per_period_rfe=False`) selects a single unified predictor subset so both periods share identical features.
- **Module 6 (`extract_features` + `run_classification`)** runs batch feature extraction (`extraction_type = 'batch'`) and trains/evaluates independent `MULTIPROBABILITY` Random Forest models for `historical` and `baseline`.
- **Module 7 (`assess_accuracy`)** performs batch thematic accuracy assessment across both `historical_val` and `baseline_val`.

In [ ]:
mt_batch = MultitemporalAnalysis(
    aoi=aoi,
    periods=periods_config,
    reference_period='baseline',
    apply_brdf=False,
    apply_topo_correction=False,
    histogram_match=True,
    scale=30,
)

#Batch is executed using Mode either via `mt_batch.run_pipeline(...)` or step-by-step by module:
# Step 1 (Module 1): Acquire imagery & build harmonized composites
mt_batch.acquire_imagery(verbose=True)

# Step 2 (Module 3): Load both training datasets -> automatically routes to `mode='batch'`
train_batch = mt_batch.load_training_data(
    training_sources={'historical': historical_roi, 'baseline': baseline_roi},
    landcover_df=scheme_df,
    class_field='ID_Kelas',
    use_class_ids=True,
    min_samples=5,
)
print('Detected training mode:', train_batch['mode'])

# Step 3 (Module 4 - Pre-Predictor): Run parametric separability (TD) on both periods independently
sep_pre_batch = mt_batch.check_separability(
    class_property='ID_Kelas',
    class_name_property='KelasTupla',
    method='TD',
    include_silhouette=False,
    use_predictors=False,
)
print('Pre-predictor separability analyzed periods:', sep_pre_batch['analyzed_periods'])
for lbl in sep_pre_batch['analyzed_periods']:
    print(f"  [{lbl}] TD summary:")
    display(sep_pre_batch['per_period'][lbl]['summary_df'])

In [ ]:
# Step 4 (Module 5): Compute unified predictors across all periods
pred_batch = mt_batch.compute_predictors(predictor_config=shared_predictor_config)
print('Computed predictor bands:', pred_batch['common_feature_bands'])

# Optional (Module 4/5 - Post-Predictor): Run non-parametric Silhouette score on selected predictors
sep_post_batch = mt_batch.check_separability(
    class_property='ID_Kelas',
    class_name_property='KelasTupla',
    method='SILHOUETTE',
    use_predictors=True,
)
for lbl in sep_post_batch['analyzed_periods']:
    sil = sep_post_batch['per_period'][lbl]['silhouette']
    print(f"  [{lbl}] Post-predictor Overall Silhouette:", sil['overall_score'])

# Step 5 (Module 6a): Batch feature extraction (independent stratified split per period)
fe_batch = mt_batch.extract_features(
    class_property='ID_Kelas',
    train_ratio=0.7,
)
print('Feature extraction type:', fe_batch['extraction_type'])

# Step 6 (Module 5b - Optional RFE_CV): Unified feature selection across periods
if shared_rfe_config and shared_rfe_config.get('enabled', True):
    rfe_batch = mt_batch.run_feature_selection(
        class_property='ID_Kelas',
        min_features=shared_rfe_config.get('min_features', 6),
        step=shared_rfe_config.get('step', 1),
        cv=shared_rfe_config.get('cv', 3),
        n_trees=shared_rfe_config.get('n_trees', 50),
        scoring=shared_rfe_config.get('scoring', 'accuracy'),
        per_period_rfe=shared_rfe_config.get('per_period_rfe', False),
        verbose=True,
    )
    print('Unified RFE_CV selected features:', rfe_batch['selected_features'])

# Step 7 (Module 6b): Batch classification (trains independent RF model per period)
cls_batch = mt_batch.run_classification(
    class_property='ID_Kelas',
    classification_type='soft',
    soft_mode='multiprobability',
    ntrees=100,
    include_log_loss=True,
    spatial_smoothing=True,
    smoothing_radius=1,
    smoothing_sigma=1.0,
)
for lbl, ev in cls_batch['model_evaluation'].items():
    print(f"  [{lbl}] Internal Test OA: {ev['overall_accuracy']:.4f} | Kappa: {ev['kappa']:.4f} | Log Loss: {ev.get('log_loss')}")

# Step 8 (Module 7): Batch accuracy assessment across both validation datasets
acc_batch = mt_batch.assess_accuracy(
    validation_by_period={'historical': historical_val, 'baseline': baseline_val},
    class_property='ID_Kelas',
)
print('Assessed validation periods:', acc_batch['assessed_periods'])
for lbl in acc_batch['assessed_periods']:
    ok, m = acc_batch['per_period'][lbl]
    if ok:
        print(f"  [{lbl}] External Validation OA: {m['overall_accuracy']:.4f} | Kappa: {m['kappa']:.4f}")

---
## 4. Post-Classification Consistency Analysis & Pre/Post Accuracy Comparison
Applies ecological transition weights $W(i \to j) \in [0, 1]$ to detect and correct illogical land-cover transitions (e.g., Built-up reverting to Primary Forest) on **both** the Model Transfer output and the Batch Mode output.

In [ ]:
ca = consistency_analysis()
class_ids = sorted(int(c) for c in scheme_df['ID'].tolist())
geom = aoi.geometry()

# Malaka Tengah scheme: 8 = Lahan Terbangun (Built-up), 9 = Lahan Terbuka
# Define forbidden transitions (weight = 0.0) and allowed transitions (weight = 1.0)
forbidden = {8: {1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 5: 0.0, 6: 0.0, 7: 0.0}, 9: {1: 0.0, 2: 0.0}}
matrix = {
    fid: {tid: (1.0 if fid == tid else forbidden.get(fid, {}).get(tid, 1.0)) for tid in class_ids}
    for fid in class_ids
}
val_mat = ca.validate_transition_matrix(matrix, class_ids=class_ids)

def run_consistency_and_compare(workflow_name, classification_dict, hist_val_fc=None):
    print(f'\n================ {workflow_name} ================')
    past_raw = classification_dict['maps']['historical']
    future_ref = classification_dict['maps']['baseline']

    ct_raw = ca.cross_tabulate(
        past_raw, future_ref, val_mat, geom, scale=30, class_names=scheme_df, class_ids=class_ids
    )
    print('Raw total px:', ct_raw['total_pixels'], '| Inconsistent px:', ct_raw['inconsistent_pixels'])
    display(ct_raw['inconsistent_transitions'])

    hmm_out = ca.correct_inconsistencies(
        past_map=past_raw,
        future_map=future_ref,
        consistency_matrix=val_mat,
        aoi=geom,
        method='hmm',
        anchor_reference=True,
        spatial_smoothing=True,
        spatial_weight=0.25,
        scale=30,
        class_ids=class_ids,
        class_names=scheme_df,
        prob_band_map=classification_dict['prob_bands']['historical'],
    )
    print('HMM Corrected px:', hmm_out['corrected_pixel_count'], '| Remaining inconsistent:', hmm_out['remaining_inconsistent_count'])

    if hist_val_fc is not None:
        taa = thematic_accuracy()
        _, raw_acc = taa.run_accuracy_assessment(past_raw, hist_val_fc, class_property='ID_Kelas', scale=30)
        _, corr_acc = taa.run_accuracy_assessment(hmm_out['corrected_map'], hist_val_fc, class_property='ID_Kelas', scale=30)
        delta = taa.compare_accuracy(raw_acc, corr_acc)
        print(f"Historical OA Before: {delta['oa_raw']:.4f} -> After HMM: {delta['oa_corrected']:.4f} (Delta OA: {delta['oa_delta']:+.4f})")
        print(f"Historical Kappa Before: {delta['kappa_raw']:.4f} -> After HMM: {delta['kappa_corrected']:.4f} (Delta Kappa: {delta['kappa_delta']:+.4f})")
    return hmm_out

hmm_transfer = run_consistency_and_compare('Workflow A: Model Transfer', res_transfer['classification'], hist_val_fc=historical_val)
hmm_batch = run_consistency_and_compare('Workflow B: Batch Mode', cls_batch, hist_val_fc=historical_val)

---
## 5. Interactive Map Comparison (Model Transfer vs. Batch Mode)

In [ ]:
Map = geemap.Map()
vis = {
    'bands': ['classification'],
    'min': int(scheme_df['ID'].min()),
    'max': int(scheme_df['ID'].max()),
    'palette': scheme_df.sort_values('ID')['Color Palette'].tolist(),
}

# Model Transfer layers
Map.addLayer(res_transfer['classification']['maps']['historical'].select('classification').clip(geom), vis, 'Transfer: 2015 Historical (Raw)')
Map.addLayer(hmm_transfer['corrected_map'].clip(geom), vis, 'Transfer: 2015 Historical (HMM Consistent)')

# Batch Mode layers
Map.addLayer(cls_batch['maps']['historical'].select('classification').clip(geom), vis, 'Batch: 2015 Historical (Raw)')
Map.addLayer(hmm_batch['corrected_map'].clip(geom), vis, 'Batch: 2015 Historical (HMM Consistent)')

# Reference 2025 Baseline layer
Map.addLayer(cls_batch['maps']['baseline'].select('classification').clip(geom), vis, '2025 Baseline')
Map.centerObject(aoi, 12)
Map